In [ ]:
import base64, hashlib, json, os, subprocess, sys, tempfile
from pathlib import Path
assets = {'runner.py': {'sha256': '4ab9116f25c88a82afdf4bf8629ce56035d7b3edd322f44f166a80625abc8111', 'base64': 'IiIiVHJ1c3RlZCBzdGVyaWxlIGVudHJ5cG9pbnQuIE5vIGFzc2Vzc29yLCBoaWRkZW4gdmFsdWVzIG9yIHByb3ZpZGVyIGNyZWRlbnRpYWxzLgpMaW51eCB4ODYtNjQgb25seS4gQWxsIGNhbmRpZGF0ZSBleGVjdXRpb24gZm9sbG93cyBzZWFsKCk7IG5vIGZhbGxiYWNrLgpQeXRob24gc3RhcnRzIGJlZm9yZSBjaHJvb3Q7IG9ubHkgZXhwbGljaXQgcnVudGltZSBmaWxlcyBzdXBwb3J0IGxhdGVyIGltcG9ydHMuCiIiIgppbXBvcnQgY3R5cGVzCmltcG9ydCBlcnJubwppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHBsYXRmb3JtCmltcG9ydCByZXNvdXJjZQppbXBvcnQgc2lnbmFsCmltcG9ydCBzb2NrZXQKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCkxJQkMgPSBjdHlwZXMuQ0RMTChOb25lLCB1c2VfZXJybm89VHJ1ZSkKIyB4ODYtNjQgbmF0aXZlIEFCSSBvbmx5OyBhbGwgdW5rbm93biBzeXNjYWxscyAoaW5jbHVkaW5nIHgzMikgcmV0dXJuIEVQRVJNLgojIE5vIHNvY2tldCwgcHJvY2VzcyBjcmVhdGlvbi9leGVjLCBwcm9jZXNzIGluc3BlY3Rpb24sIHNpZ25hbHMsIGZpbGVzeXN0ZW0KIyBtdXRhdGlvbiwgbmFtZXNwYWNlcywgcHJpdmlsZWdlIGNoYW5nZXMsIGlvX3VyaW5nLCBicGYgb3Iga2V5cmluZyBjYWxscy4KQUxMT1dFRCA9IHswLCAxLCAyLCAzLCA1LCA4LCA5LCAxMCwgMTEsIDEyLCAxMywgMTQsIDE1LCAxNywgMTksIDIwLAogICAgICAgICAgIDIxLCAyNCwgMjUsIDI4LCAzNSwgMzksIDYwLCA2MywgNzksIDg5LCA5NywgMTAyLCAxMDQsCiAgICAgICAgICAgMTA3LCAxMDgsIDExMCwgMTE1LCAxMTgsIDEyMCwgMTU4LCAxODYsIDIwMiwgMjE4LAogICAgICAgICAgIDIyOCwgMjI5LCAyMzAsIDIzMSwgMjU3LCAyNjIsIDI2NywgMjczLCAzMTgsIDMzMn0KCmNsYXNzIEZpbHRlcihjdHlwZXMuU3RydWN0dXJlKToKICAgIF9maWVsZHNfID0gWygnY29kZScsIGN0eXBlcy5jX3VzaG9ydCksICgnanQnLCBjdHlwZXMuY191Ynl0ZSksCiAgICAgICAgICAgICAgICAoJ2pmJywgY3R5cGVzLmNfdWJ5dGUpLCAoJ2snLCBjdHlwZXMuY191aW50KV0KY2xhc3MgUHJvZ3JhbShjdHlwZXMuU3RydWN0dXJlKToKICAgIF9maWVsZHNfID0gWygnbGVuZ3RoJywgY3R5cGVzLmNfdXNob3J0KSwgKCdmaWx0ZXJzJywgY3R5cGVzLlBPSU5URVIoRmlsdGVyKSldCmNsYXNzIENhcEhlYWRlcihjdHlwZXMuU3RydWN0dXJlKToKICAgIF9maWVsZHNfID0gWygndmVyc2lvbicsIGN0eXBlcy5jX3VpbnQpLCAoJ3BpZCcsIGN0eXBlcy5jX2ludCldCmNsYXNzIENhcERhdGEoY3R5cGVzLlN0cnVjdHVyZSk6CiAgICBfZmllbGRzXyA9IFsoJ2VmZmVjdGl2ZScsIGN0eXBlcy5jX3VpbnQpLCAoJ3Blcm1pdHRlZCcsIGN0eXBlcy5jX3VpbnQpLAogICAgICAgICAgICAgICAgKCdpbmhlcml0YWJsZScsIGN0eXBlcy5jX3VpbnQpXQoKZGVmIHByY3RsKG9wdGlvbiwgYT0wLCBiPTAsIGM9MCwgZD0wKToKICAgIHJlc3VsdCA9IExJQkMucHJjdGwoY3R5cGVzLmNfaW50KG9wdGlvbiksIGN0eXBlcy5jX3Vsb25nKGEpLAogICAgICAgICAgICAgICAgICAgICAgICBjdHlwZXMuY191bG9uZyhiKSwgY3R5cGVzLmNfdWxvbmcoYyksIGN0eXBlcy5jX3Vsb25nKGQpKQogICAgaWYgcmVzdWx0IDwgMDoKICAgICAgICByYWlzZSBPU0Vycm9yKGN0eXBlcy5nZXRfZXJybm8oKSwgJ3ByY3RsICVzJyAlIG9wdGlvbikKICAgIHJldHVybiByZXN1bHQKCmRlZiBxdWFsaWZ5KCk6CiAgICBpZiBzeXMucGxhdGZvcm0gIT0gJ2xpbnV4JyBvciBwbGF0Zm9ybS5tYWNoaW5lKCkgIT0gJ3g4Nl82NCcgb3IgY3R5cGVzLnNpemVvZihjdHlwZXMuY192b2lkX3ApICE9IDg6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdVTlNVUFBPUlRFRF9QTEFURk9STScpCiAgICBpZiBvcy5nZXRldWlkKCkgIT0gMDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ1JPT1RfUkVRVUlSRUQnKQogICAgaWYgbm90IGhhc2F0dHIob3MsICdjaHJvb3QnKSBvciBub3QgaGFzYXR0cihvcywgJ3NldHJlc3VpZCcpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignTUlTU0lOR19JREVOVElUWV9QUklNSVRJVkUnKQoKZGVmIHNlY2NvbXAoKToKICAgIGluc3RydWN0aW9ucyA9IFsoMHgyMCwgMCwgMCwgNCksICgweDE1LCAxLCAwLCAweEMwMDAwMDNFKSwKICAgICAgICAgICAgICAgICAgICAoMHgwNiwgMCwgMCwgMHg4MDAwMDAwMCksICgweDIwLCAwLCAwLCAwKV0KICAgIGZvciBudW1iZXIgaW4gc29ydGVkKEFMTE9XRUQpOgogICAgICAgIGluc3RydWN0aW9ucyArPSBbKDB4MTUsIDAsIDEsIG51bWJlciksICgweDA2LCAwLCAwLCAweDdGRkYwMDAwKV0KICAgIGluc3RydWN0aW9ucyArPSBbKDB4MDYsIDAsIDAsIDB4MDAwNTAwMDAgfCBlcnJuby5FUEVSTSldCiAgICBmaWx0ZXJzID0gKEZpbHRlciAqIGxlbihpbnN0cnVjdGlvbnMpKSgqKEZpbHRlcigqeCkgZm9yIHggaW4gaW5zdHJ1Y3Rpb25zKSkKICAgIHByb2dyYW0gPSBQcm9ncmFtKGxlbihpbnN0cnVjdGlvbnMpLCBmaWx0ZXJzKQogICAgcHJjdGwoMzgsIDEpCiAgICBpZiBwcmN0bCgzOSkgIT0gMToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ05PX05FV19QUklWU19OT1RfU0VUJykKICAgIGlmIExJQkMucHJjdGwoMjIsIDIsIGN0eXBlcy5ieXJlZihwcm9ncmFtKSwgMCwgMCkgIT0gMDoKICAgICAgICByYWlzZSBPU0Vycm9yKGN0eXBlcy5nZXRfZXJybm8oKSwgJ3NlY2NvbXAgaW5zdGFsbCcpCgpkZWYgYXJtX3BhcmVudF9kZWF0aChwYXJlbnRfcGlkKToKICAgIHByY3RsKDEsIHNpZ25hbC5TSUdLSUxMKQogICAgb2JzZXJ2ZWQgPSBjdHlwZXMuY19pbnQoKQogICAgaWYgTElCQy5wcmN0bCgyLCBjdHlwZXMuYnlyZWYob2JzZXJ2ZWQpLCAwLCAwLCAwKSAhPSAwIG9yIG9ic2VydmVkLnZhbHVlICE9IHNpZ25hbC5TSUdLSUxMOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignUERFQVRIU0lHX05PVF9TRVQnKQogICAgaWYgb3MuZ2V0cHBpZCgpICE9IHBhcmVudF9waWQ6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdQQVJFTlRfQ0hBTkdFRCcpCgoKZGVmIHNlYWwocm9vdCwgcGFyZW50X3BpZD1Ob25lKToKICAgIHF1YWxpZnkoKQogICAgcGFyZW50X3BpZCA9IG9zLmdldHBwaWQoKSBpZiBwYXJlbnRfcGlkIGlzIE5vbmUgZWxzZSBwYXJlbnRfcGlkCiAgICBhcm1fcGFyZW50X2RlYXRoKHBhcmVudF9waWQpCiAgICAjIFRoaXMgZnJlc2ggZXhlYyBoYXMgcmVjZWl2ZWQgbm8gYXNzZXNzb3IgZGF0YTsgcHJlLW9wZW5lZCBjb2RlL2FyZ3Mgb25seS4KICAgICMgQ2xvc2UgZXZlcnkgZXh0cmEgZGVzY3JpcHRvciwgbm90IG1lcmVseSB0aG9zZSBiZWxvdyBhIGxvd2VyZWQgcmxpbWl0LgogICAgZmRzID0gW2ludChwLm5hbWUpIGZvciBwIGluIFBhdGgoJy9wcm9jL3NlbGYvZmQnKS5pdGVyZGlyKCldCiAgICBudWxsID0gb3Mub3BlbignL2Rldi9udWxsJywgb3MuT19SRE9OTFkpCiAgICBvcy5kdXAyKG51bGwsIDApCiAgICBvcy5jbG9zZShudWxsKQogICAgZm9yIGZkIGluIGZkczoKICAgICAgICBpZiBmZCA+IDI6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIG9zLmNsb3NlKGZkKQogICAgICAgICAgICBleGNlcHQgT1NFcnJvciBhcyBlcnJvcjoKICAgICAgICAgICAgICAgIGlmIGVycm9yLmVycm5vICE9IGVycm5vLkVCQURGOgogICAgICAgICAgICAgICAgICAgIHJhaXNlCiAgICBvcy5lbnZpcm9uLmNsZWFyKCkKICAgIG9zLmVudmlyb24udXBkYXRlKHsnTEFORyc6ICdDJywgJ1BZVEhPTkRPTlRXUklURUJZVEVDT0RFJzogJzEnfSkKICAgIG9zLmNocm9vdChyb290KQogICAgb3MuY2hkaXIoJy8nKQogICAgc3lzLnBhdGhbOl0gPSBbJy9ydW50aW1lJ10KICAgIHN5cy5hcmd2WzpdID0gWydjYW5kaWRhdGUnXQogICAgIyBEcm9wIGFtYmllbnQgYW5kIEVWRVJZIHN1cHBvcnRlZCBib3VuZGluZyBjYXBhYmlsaXR5IHdoaWxlIHByaXZpbGVnZWQuCiAgICBwcmN0bCg0NywgNCkKICAgIGNhcCA9IDAKICAgIHdoaWxlIFRydWU6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBwcmN0bCgyMywgY2FwKQogICAgICAgIGV4Y2VwdCBPU0Vycm9yIGFzIGVycm9yOgogICAgICAgICAgICBpZiBlcnJvci5lcnJubyA9PSBlcnJuby5FSU5WQUw6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICByYWlzZQogICAgICAgIHByY3RsKDI0LCBjYXApCiAgICAgICAgaWYgcHJjdGwoMjMsIGNhcCkgIT0gMDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdCT1VORElOR19DQVBfUkVNQUlOUycpCiAgICAgICAgaWYgcHJjdGwoNDcsIDEsIGNhcCkgIT0gMDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdBTUJJRU5UX0NBUF9SRU1BSU5TJykKICAgICAgICBjYXAgKz0gMQogICAgICAgIGlmIGNhcCA+IDEyODoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdVTktOT1dOX0NBUF9SQU5HRScpCiAgICBvcy5zZXRncm91cHMoW10pCiAgICBvcy5zZXRyZXNnaWQoNjU1MzQsIDY1NTM0LCA2NTUzNCkKICAgIG9zLnNldHJlc3VpZCg2NTUzNCwgNjU1MzQsIDY1NTM0KQogICAgaGVhZGVyID0gQ2FwSGVhZGVyKDB4MjAwODA1MjIsIDApCiAgICBkYXRhID0gKENhcERhdGEgKiAyKSgpCiAgICBpZiBMSUJDLmNhcHNldChjdHlwZXMuYnlyZWYoaGVhZGVyKSwgZGF0YSkgIT0gMDoKICAgICAgICByYWlzZSBPU0Vycm9yKGN0eXBlcy5nZXRfZXJybm8oKSwgJ2NhcHNldCcpCiAgICBpZiBMSUJDLmNhcGdldChjdHlwZXMuYnlyZWYoaGVhZGVyKSwgZGF0YSkgIT0gMCBvciBhbnkoeC5lZmZlY3RpdmUgb3IgeC5wZXJtaXR0ZWQgb3IgeC5pbmhlcml0YWJsZSBmb3IgeCBpbiBkYXRhKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0NBUFNfUkVNQUlOJykKICAgIGlmIG9zLmdldHJlc3VpZCgpICE9ICg2NTUzNCwpKjMgb3Igb3MuZ2V0cmVzZ2lkKCkgIT0gKDY1NTM0LCkqMyBvciBvcy5nZXRncm91cHMoKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0lERU5USVRZX1JFTUFJTlMnKQogICAgIyBJZGVudGl0eSB0cmFuc2l0aW9ucyBjbGVhciBQREVBVEhTSUcuIFJlLWFybSBhbmQgdmVyaWZ5IHRoZSBvcmlnaW5hbAogICAgIyBwYXJlbnQgaXMgc3RpbGwgYWxpdmUgYmVmb3JlIHRoZSBjYW5kaWRhdGUgY2FuIHJ1bi4KICAgIGFybV9wYXJlbnRfZGVhdGgocGFyZW50X3BpZCkKICAgIGZvciBuYW1lLCB2YWx1ZSBpbiBbKCdSTElNSVRfQVMnLCAyNjg0MzU0NTYpLCAoJ1JMSU1JVF9DUFUnLCAyKSwKICAgICAgICAgICAgICAgICAgICAgICAgKCdSTElNSVRfRlNJWkUnLCA2NTUzNiksICgnUkxJTUlUX0NPUkUnLCAwKSwKICAgICAgICAgICAgICAgICAgICAgICAgKCdSTElNSVRfTk9GSUxFJywgMzIpLCAoJ1JMSU1JVF9OUFJPQycsIDApXToKICAgICAgICByZXNvdXJjZS5zZXRybGltaXQoZ2V0YXR0cihyZXNvdXJjZSwgbmFtZSksICh2YWx1ZSwgdmFsdWUpKQogICAgcmVjZWlwdCA9IHsndWlkJzogbGlzdChvcy5nZXRyZXN1aWQoKSksICdnaWQnOiBsaXN0KG9zLmdldHJlc2dpZCgpKSwKICAgICAgICAgICAgICAgJ2dyb3Vwcyc6IG9zLmdldGdyb3VwcygpLCAnY2FwYWJpbGl0aWVzX3plcm8nOiBUcnVlLAogICAgICAgICAgICAgICAnYm91bmRpbmdfZHJvcHBlZCc6IGNhcCwgJ2FtYmllbnRfY2xlYXJlZCc6IFRydWUsCiAgICAgICAgICAgICAgICdwYXJlbnRfZGVhdGhfc2lnbmFsJzogc2lnbmFsLlNJR0tJTEwsCiAgICAgICAgICAgICAgICdjd2QnOiBvcy5nZXRjd2QoKSwgJ25vX25ld19wcml2cyc6IFRydWUsICdzZWNjb21wJzogJ25hdGl2ZS1hbGxvd2xpc3QnfQogICAgc2VjY29tcCgpCiAgICAjIFRydXN0ZWQgc3RhcnR1cCByZWNlaXB0IHByZWNlZGVzIHRoZSB1bnRydXN0ZWQgY29kZS4gQXNzZXNzb3IgbmV2ZXIgdHJlYXRzCiAgICAjIGFueSBsYXRlciBzdGRvdXQgYXMgYSB0cnVzdGVkIHZlcmRpY3Q7IG91dHB1dHMgcmVxdWlyZSBleGFjdCBjYXNlIG1hdGNoZXMuCiAgICBvcy53cml0ZSgyLCAoanNvbi5kdW1wcyh7J3NlYWxlZCc6IHJlY2VpcHR9KSsnXG4nKS5lbmNvZGUoKSkKCmRlZiBtYWluKCk6CiAgICBpZiBsZW4oc3lzLmFyZ3YpICE9IDMgb3Igbm90IHN5cy5hcmd2WzJdLmlzZGlnaXQoKSBvciBpbnQoc3lzLmFyZ3ZbMl0pIDw9IDE6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdST09UX0FORF9FWFBFQ1RFRF9QQVJFTlRfUkVRVUlSRUQnKQogICAgcGFyZW50X3BpZCA9IGludChzeXMuYXJndlsyXSkKICAgIGFybV9wYXJlbnRfZGVhdGgocGFyZW50X3BpZCkKICAgIHJvb3QgPSBQYXRoKHN5cy5hcmd2WzFdKS5yZXNvbHZlKHN0cmljdD1UcnVlKQogICAgc291cmNlID0gKHJvb3QvJ3dvcmsvY2FuZGlkYXRlLnB5JykucmVhZF90ZXh0KCkKICAgIHJlcXVlc3QgPSBqc29uLmxvYWRzKChyb290Lyd3b3JrL2FyZ3MuanNvbicpLnJlYWRfdGV4dCgpKQogICAgaWYgc2V0KHJlcXVlc3QpICE9IHsnbW9kZScsICdhcmdzJ30gb3IgcmVxdWVzdFsnbW9kZSddIG5vdCBpbiAoJ2NhbmRpZGF0ZScsICdpbnNwZWN0aW9uJykgb3Igbm90IGlzaW5zdGFuY2UocmVxdWVzdFsnYXJncyddLCBsaXN0KToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0JBRF9QUk9UT0NPTCcpCiAgICBzZWFsKHN0cihyb290KSwgcGFyZW50X3BpZCkKICAgIGNvZGUgPSBjb21waWxlKHNvdXJjZSwgJy93b3JrL2NhbmRpZGF0ZS5weScsICdleGVjJykKICAgICMgU291cmNlL3JlcXVlc3QgY29udGFpbiBvbmx5IHRoZSBjdXJyZW50IGNhbGwsIG5ldmVyIHRoZSBhc3Nlc3NvciBmaXh0dXJlLgogICAgc2NvcGUgPSB7J19fbmFtZV9fJzogJ19fY2FuZGlkYXRlX18nfQogICAgZXhlYyhjb2RlLCBzY29wZSwgc2NvcGUpCiAgICB2YWx1ZSA9IHNjb3BlWydydW4nXSgqcmVxdWVzdFsnYXJncyddKQogICAgcHJpbnQoanNvbi5kdW1wcyh2YWx1ZSwgYWxsb3dfbmFuPUZhbHNlKSwgZmx1c2g9VHJ1ZSkKCmlmIF9fbmFtZV9fID09ICdfX21haW5fXyc6CiAgICBtYWluKCkK'}, 'probe.py': {'sha256': 'bf737e70dfcf4069dc9b71916198f0eb66365a7d5fd8592715e71337ec8c6f51', 'base64': '...'}, 'accelerator.py': {'sha256': 'e291f21804896ca6d110749352b484b7a65c4f75f67b79efd2fcdf55caaf5916', 'base64': 'IiIiVHJ1c3RlZC1vbmx5IHJ1bnRpbWUgZGlhZ25vc3RpY3MuIE5ldmVyIHVzZWQgYXMgYSBjYW5kaWRhdGUgZW52aXJvbm1lbnQuIiIiCmltcG9ydCBjc3YKaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgcmUKaW1wb3J0IHN5cwoKQklOQVJZX1BBVEhTID0gKCcvb3B0L2Jpbi9udmlkaWEtc21pJywgJy91c3IvYmluL252aWRpYS1zbWknLCAnL3Vzci9sb2NhbC9iaW4vbnZpZGlhLXNtaScsICcvYmluL252aWRpYS1zbWknKQpMSUJSQVJZX1BBVEhTID0gKCcvdXNyL2xvY2FsL252aWRpYS9saWI2NCcsICcvdXNyL2xvY2FsL252aWRpYS9saWInLCAnL3Vzci9saWIveDg2XzY0LWxpbnV4LWdudScsICcvdXNyL2xpYjY0JykKQkFTRV9FTlYgPSB7J1BBVEgnOiAnL3Vzci9sb2NhbC9iaW46L3Vzci9iaW46L2JpbicsICdMQU5HJzogJ0MnfQoKCmRlZiBsb2FkZXJfZW52aXJvbm1lbnQocGF0aHM9TElCUkFSWV9QQVRIUyk6CiAgICBldmlkZW5jZSwgYWNjZXB0ZWQgPSBbXSwgW10KICAgIGZvciBuYW1lIGluIHBhdGhzOgogICAgICAgIHBhdGggPSBQYXRoKG5hbWUpCiAgICAgICAgcm93ID0geydwYXRoJzogbmFtZSwgJ2FjY2VwdGVkJzogRmFsc2UsICdsaWJyYXJpZXMnOiBbXX0KICAgICAgICBldmlkZW5jZS5hcHBlbmQocm93KQogICAgICAgIHRyeToKICAgICAgICAgICAgcmVzb2x2ZWQgPSBwYXRoLnJlc29sdmUoc3RyaWN0PVRydWUpCiAgICAgICAgICAgIGlmIG5vdCByZXNvbHZlZC5pc19kaXIoKSBvciBhbnkocC5zdGF0KCkuc3RfdWlkICE9IDAgb3IgcC5zdGF0KCkuc3RfbW9kZSAmIDBvMDIyIGZvciBwIGluIChyZXNvbHZlZCwgKnJlc29sdmVkLnBhcmVudHMpKToKICAgICAgICAgICAgICAgIHJvd1snZXJyb3InXSA9ICdVTlRSVVNURURfTElCUkFSWV9ESVJFQ1RPUlknCiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBmb3IgcGF0dGVybiBpbiAoJ2xpYm52aWRpYS1tbC5zbyonLCAnbGliY3VkYS5zbyonKToKICAgICAgICAgICAgICAgIGZvciBsaWIgaW4gc29ydGVkKHBhdGguZ2xvYihwYXR0ZXJuKSk6CiAgICAgICAgICAgICAgICAgICAgdGFyZ2V0ID0gbGliLnJlc29sdmUoc3RyaWN0PVRydWUpCiAgICAgICAgICAgICAgICAgICAgcm93WydsaWJyYXJpZXMnXS5hcHBlbmQoeyduYW1lJzogbGliLm5hbWUsICdyZXNvbHZlZCc6IHN0cih0YXJnZXQpLCAnc2l6ZSc6IHRhcmdldC5zdGF0KCkuc3Rfc2l6ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgJ3Jvb3Rfb3duZWQnOiB0YXJnZXQuc3RhdCgpLnN0X3VpZCA9PSAwLCAnd3JpdGFibGUnOiBib29sKHRhcmdldC5zdGF0KCkuc3RfbW9kZSAmIDBvMDIyKX0pCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IHRhcmdldC5pc19maWxlKCkgb3IgdGFyZ2V0LnN0YXQoKS5zdF91aWQgIT0gMCBvciB0YXJnZXQuc3RhdCgpLnN0X21vZGUgJiAwbzAyMjoKICAgICAgICAgICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdVTlRSVVNURURfTElCUkFSWV9GSUxFJykKICAgICAgICAgICAgYWNjZXB0ZWQuYXBwZW5kKG5hbWUpCiAgICAgICAgICAgIHJvd1snYWNjZXB0ZWQnXSA9IFRydWUKICAgICAgICBleGNlcHQgKE9TRXJyb3IsIFJ1bnRpbWVFcnJvcikgYXMgZXJyb3I6CiAgICAgICAgICAgIHJvd1snZXJyb3InXSA9IHR5cGUoZXJyb3IpLl9fbmFtZV9fICsgJzonICsgc3RyKGVycm9yKQogICAgZW52ID0gZGljdChCQVNFX0VOVikKICAgIGlmIGFjY2VwdGVkOgogICAgICAgIGVudlsnTERfTElCUkFSWV9QQVRIJ10gPSAnOicuam9pbihhY2NlcHRlZCkKICAgIHJldHVybiBlbnYsIGV2aWRlbmNlCgoKZGVmIHBhcnNlX2RldmljZShyZWNlaXB0KToKICAgIGlmIG5vdCByZWNlaXB0IG9yIHJlY2VpcHQuZ2V0KCdzdGF0dXMnLCAnY29tcGxldGVkJykgIT0gJ2NvbXBsZXRlZCcgb3IgcmVjZWlwdC5nZXQoJ3JldHVybmNvZGUnKSAhPSAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0FDQ0VMRVJBVE9SX1FVRVJZX0ZBSUxFRCcpCiAgICByb3dzID0gbGlzdChjc3YucmVhZGVyKHJlY2VpcHRbJ3N0ZG91dCddLnNwbGl0bGluZXMoKSwgc2tpcGluaXRpYWxzcGFjZT1UcnVlKSkKICAgIGlmIGxlbihyb3dzKSAhPSAxIG9yIGxlbihyb3dzWzBdKSAhPSA0OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0FDQ0VMRVJBVE9SX1FVRVJZX1NIQVBFJykKICAgIG5hbWUsIHV1aWQsIGRyaXZlciwgbWVtb3J5ID0gKHMuc3RyaXAoKSBmb3IgcyBpbiByb3dzWzBdKQogICAgaWYgbm90IHJlLmZ1bGxtYXRjaChyJyg/Ok5WSURJQSApP1JUWCBQUk8gNjAwMCg/OiBCbGFja3dlbGwoPzogU2VydmVyfCBXb3Jrc3RhdGlvbik/IEVkaXRpb24pPycsIG5hbWUsIHJlLkkpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0FDQ0VMRVJBVE9SX05BTUVfTUlTTUFUQ0gnKQogICAgaWYgbm90IHJlLmZ1bGxtYXRjaChyJ0dQVS1bMC05YS1mQS1GXXs4fSg/Oi1bMC05YS1mQS1GXXs0fSl7M30tWzAtOWEtZkEtRl17MTJ9JywgdXVpZCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignQUNDRUxFUkFUT1JfVVVJRF9NQUxGT1JNRUQnKQogICAgaWYgbm90IHJlLmZ1bGxtYXRjaChyJ1xkKyg/OlwuXGQrKSsnLCBkcml2ZXIpIG9yIG5vdCByZS5mdWxsbWF0Y2gocidbMS05XVxkKiBNaUInLCBtZW1vcnkpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0FDQ0VMRVJBVE9SX0ZJRUxEU19NQUxGT1JNRUQnKQogICAgcmV0dXJuIHsnbmFtZSc6IG5hbWUsICd1dWlkJzogdXVpZCwgJ2RyaXZlcl92ZXJzaW9uJzogZHJpdmVyLCAnbWVtb3J5X21pYic6IGludChtZW1vcnkuc3BsaXQoKVswXSksICdkZXZpY2VfY291bnQnOiAxfQoKCmRlZiBkaXNjb3ZlcihleGVjdXRlLCBwYXRocz1CSU5BUllfUEFUSFMpOgogICAgZW52LCBsaWJyYXJpZXMgPSBsb2FkZXJfZW52aXJvbm1lbnQoKQogICAgYXR0ZW1wdHMgPSBbXQogICAgcmVzdWx0ID0geydyZXR1cm5jb2RlJzogTm9uZSwgJ3N0ZG91dCc6ICcnLCAnc3RkZXJyJzogJycsICdhdHRlbXB0cyc6IGF0dGVtcHRzLAogICAgICAgICAgICAgICdsb2FkZXJfZGlyZWN0b3JpZXMnOiBsaWJyYXJpZXMsICd0cnVzdGVkX2Vudmlyb25tZW50JzogZW52LCAnZGVwZW5kZW5jeV9jaGVja3MnOiB7fX0KICAgICMgSW5kZXBlbmRlbnQgZGlhZ25vc3RpY3M6IGEgcXVlcnkgZmFpbHVyZSBtdXN0IG5vdCBzdXBwcmVzcyBkcml2ZXIgbG9hZGluZyBmYWN0cy4KICAgIGZvciBsaWJyYXJ5IGluICgnbGlibnZpZGlhLW1sLnNvLjEnLCAnbGliY3VkYS5zby4xJyk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICByZXN1bHRbJ2RlcGVuZGVuY3lfY2hlY2tzJ11bbGlicmFyeV0gPSBleGVjdXRlKFtzeXMuZXhlY3V0YWJsZSwgJy1JJywgJy1TJywgJy1jJywKICAgICAgICAgICAgICAgICdpbXBvcnQgY3R5cGVzOyBjdHlwZXMuQ0RMTCgnICsgcmVwcihsaWJyYXJ5KSArICcpOyBwcmludCgiTE9BRF9PSyIpJ10sIHRpbWVvdXQ9NSwgb3V0cHV0X2NhcD00MDk2LCBlbnY9ZW52KQogICAgICAgIGV4Y2VwdCBPU0Vycm9yIGFzIGVycm9yOgogICAgICAgICAgICByZXN1bHRbJ2RlcGVuZGVuY3lfY2hlY2tzJ11bbGlicmFyeV0gPSB7J2Vycm9yJzogdHlwZShlcnJvcikuX19uYW1lX18gKyAnOicgKyBzdHIoZXJyb3IpfQogICAgZm9yIG5hbWUgaW4gcGF0aHM6CiAgICAgICAgYXR0ZW1wdCA9IHsncGF0aCc6IG5hbWV9CiAgICAgICAgYXR0ZW1wdHMuYXBwZW5kKGF0dGVtcHQpCiAgICAgICAgdHJ5OgogICAgICAgICAgICBpZiBub3QgUGF0aChuYW1lKS5pc19maWxlKCkgb3Igbm90IG9zLmFjY2VzcyhuYW1lLCBvcy5YX09LKToKICAgICAgICAgICAgICAgIGF0dGVtcHRbJ2Vycm9yJ10gPSAnTk9UX1VTQUJMRV9GSUxFJwogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgYXR0ZW1wdC51cGRhdGUoZXhlY3V0ZShbbmFtZSwgJy0tcXVlcnktZ3B1PW5hbWUsdXVpZCxkcml2ZXJfdmVyc2lvbixtZW1vcnkudG90YWwnLCAnLS1mb3JtYXQ9Y3N2LG5vaGVhZGVyJ10sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdGltZW91dD0xMCwgb3V0cHV0X2NhcD04MTkyLCBlbnY9ZW52KSkKICAgICAgICBleGNlcHQgT1NFcnJvciBhcyBlcnJvcjoKICAgICAgICAgICAgYXR0ZW1wdFsnZXJyb3InXSA9IHR5cGUoZXJyb3IpLl9fbmFtZV9fICsgJzonICsgc3RyKGVycm9yKQogICAgICAgIHJlc3VsdC51cGRhdGUoYXR0ZW1wdCkKICAgICAgICB0cnk6CiAgICAgICAgICAgIHJlc3VsdFsnb2JzZXJ2ZWRfZGV2aWNlJ10gPSBwYXJzZV9kZXZpY2UocmVzdWx0KQogICAgICAgIGV4Y2VwdCAoVmFsdWVFcnJvciwgS2V5RXJyb3IpIGFzIGVycm9yOgogICAgICAgICAgICByZXN1bHRbJ3ZhbGlkYXRpb25fZXJyb3InXSA9IHN0cihlcnJvcikKICAgICAgICByZXR1cm4gcmVzdWx0CiAgICByZXN1bHRbJ2Vycm9yJ10gPSAnTlZJRElBX1NNSV9VTkFWQUlMQUJMRScKICAgIHJldHVybiByZXN1bHQK'}, 'audit_remote.py': {'sha256': '7bdc02cadd4c5d91cb5700d0461e9355bdc771324a315c87215f664da4a69a62', 'base64': 'IiIiRmFpbC1jbG9zZWQgcmVjZWlwdCBhdWRpdC4gTG9jYWwgZml4dHVyZXMgbmV2ZXIgZXN0YWJsaXNoIHJlbW90ZSBwcm92ZW5hbmNlLiIiIgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHJlCmltcG9ydCBzeXMKaW1wb3J0IGFjY2VsZXJhdG9yCgpOQU1FUyA9ICgnYmVuaWduJywnZmlsZXN5c3RlbScsJ2luaGVyaXRlZCcsJ3N5c2NhbGxzJywncmVhZG9ubHknLCd3YWxsJywnb3V0cHV0JywnZm9yZ2VkJykKREVOSUVEID0gJ3NvY2tldCBjb25uZWN0IHB0cmFjZSBwcm9jZXNzX3ZtX3JlYWR2IHBpZGZkX2dldGZkIG1vdW50IGNocm9vdCBzZXRucyB1bnNoYXJlIHNldHVpZCBzZXRnaWQgY2Fwc2V0IHByY3RsIGNsb25lIGZvcmsgdmZvcmsgY2xvbmUzIGV4ZWN2ZSBleGVjdmVhdCBpb191cmluZ19zZXR1cCBvcGVuX2J5X2hhbmRsZV9hdCBwcmxpbWl0NjQgZmNudGwgc2NoZWRfZ2V0YWZmaW5pdHkgeDMyX2dldHBpZCcuc3BsaXQoKQpRVUFMSUZJRUQgPSAnUVVBTElGSUVEX1NZTlRIRVRJQ19QUk9CRV9PTkxZJwpDTEFTU0lGSUNBVElPTlMgPSB7UVVBTElGSUVELCAnVU5TVVBQT1JURURfT1JfUVVBTElGSUNBVElPTl9GQUlMRUQnLCAnQ0xFQU5VUF9GQUlMRUQnLCAnSU5DT01QTEVURV9RVUFMSUZJQ0FUSU9OJ30KCgpkZWYgcmVxdWlyZShvaywgcmVhc29uKToKICAgIGlmIG5vdCBvazoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdBVURJVDonICsgcmVhc29uKQoKCmRlZiBwcm92ZW5hbmNlKHN0YXR1cywgc2VydmVyLCAqLCBrZXJuZWwsIHZlcnNpb24sIGltYWdlLCBleHBlY3RlZF9ub3RlYm9va19zaGEyNTYsIG5vdGVib29rKToKICAgIHJlcXVpcmUodHlwZSh2ZXJzaW9uKSBpcyBpbnQgYW5kIHZlcnNpb24gPiAwLCAnRVhBQ1RfVkVSU0lPTl9SRVFVSVJFRCcpCiAgICB0ZXJtaW5hbCA9IHN0YXR1cy5nZXQoJ3N0YXR1cycpCiAgICByZXF1aXJlKHN0YXR1cy5nZXQoJ3JldHVybmNvZGUnKSA9PSAwIGFuZCB0ZXJtaW5hbCBpbiAoJ0NPTVBMRVRFJywnRVJST1InLCdDQU5DRUxMRUQnLCdDQU5DRUxFRCcpLCAnU1RBVFVTX1FVRVJZX0ZBSUxFRF9PUl9OT05URVJNSU5BTCcpCiAgICByZXF1aXJlKHN0YXR1cy5nZXQoJ3N0ZG91dCcsICcnKS5zdHJpcCgpID09IGYne2tlcm5lbH0ve3ZlcnNpb259IGhhcyBzdGF0dXMgIktlcm5lbFdvcmtlclN0YXR1cy57dGVybWluYWx9IicsICdTVEFUVVNfVEFSR0VUX1ZFUlNJT05fTUlTTUFUQ0gnKQogICAgcmVxdWlyZShzZXJ2ZXIuZ2V0KCdpZCcpID09IGtlcm5lbCBhbmQgc2VydmVyLmdldCgnY3VycmVudF92ZXJzaW9uX251bWJlcicpID09IHZlcnNpb24sICdTRVJWRVJfVEFSR0VUX1ZFUlNJT05fTUlTTUFUQ0gnKQogICAgcmVxdWlyZShzZXJ2ZXIuZ2V0KCdpc19wcml2YXRlJykgaXMgVHJ1ZSBhbmQgc2VydmVyLmdldCgnZW5hYmxlX2ludGVybmV0JykgaXMgRmFsc2UsICdTRVJWRVJfUFJJVkFDWV9ORVRXT1JLJykKICAgIHJlcXVpcmUoc2VydmVyLmdldCgnbWFjaGluZV9zaGFwZScpID09ICdOdmlkaWFSdHhQcm82MDAwJyBhbmQgc2VydmVyLmdldCgnZG9ja2VyX2ltYWdlJykgPT0gaW1hZ2UsICdTRVJWRVJfUkVTT1VSQ0VfSU1BR0UnKQogICAgcmVxdWlyZShhbGwoc2VydmVyLmdldChrKSA9PSBbXSBmb3IgayBpbiAoJ2RhdGFzZXRfc291cmNlcycsJ2tlcm5lbF9zb3VyY2VzJywnY29tcGV0aXRpb25fc291cmNlcycsJ21vZGVsX3NvdXJjZXMnKSksICdTRVJWRVJfU09VUkNFUycpCiAgICByZXF1aXJlKGhhc2hsaWIuc2hhMjU2KG5vdGVib29rKS5oZXhkaWdlc3QoKSA9PSBleHBlY3RlZF9ub3RlYm9va19zaGEyNTYsICdTRVJWRVJfTk9URUJPT0tfRFJJRlQnKQogICAgcmV0dXJuIHsna2VybmVsJzoga2VybmVsLCAndmVyc2lvbic6IHZlcnNpb24sICd0ZXJtaW5hbF9zdGF0dXMnOiB0ZXJtaW5hbCwgJ3Byb3ZlbmFuY2UnOiAnRVhBQ1RfVEFSR0VUX1JFQ09OQ0lMRUQnfQoKCmRlZiBhdWRpdF9kaXJlY3RvcnkoZGlyZWN0b3J5LCAqLCBpZGVudGl0eSwgc291cmNlcywgbG9jYWxfZml4dHVyZT1GYWxzZSk6CiAgICBkaXJlY3RvcnkgPSBQYXRoKGRpcmVjdG9yeSkKICAgIHJlc3VsdCA9IGpzb24ubG9hZHMoKGRpcmVjdG9yeS8ncmVzdWx0Lmpzb24nKS5yZWFkX3RleHQoKSkKICAgIHJlcXVpcmUocmVzdWx0LmdldCgnaWRlbnRpdHknKSA9PSBpZGVudGl0eSBhbmQgcmVzdWx0LmdldCgnc291cmNlX3NoYTI1NicpID09IHNvdXJjZXMsICdSRVNVTFRfU09VUkNFX0lERU5USVRZJykKICAgIHJlcXVpcmUocmVzdWx0LmdldCgnY2xhc3NpZmljYXRpb24nKSBpbiBDTEFTU0lGSUNBVElPTlMsICdVTktOT1dOX0NMQVNTSUZJQ0FUSU9OJykKICAgIHJlcXVpcmUocmVzdWx0LmdldCgnbW9kZWxfbG9hZGVkJykgaXMgRmFsc2UsICdNT0RFTF9MT0FERUQnKQogICAgcm93cyA9IHJlc3VsdFsnY2FzZXMnXQogICAgcmVxdWlyZShpc2luc3RhbmNlKHJvd3MsIGxpc3QpLCAnQ0FTRV9ST1dTJykKICAgIHJlcXVpcmVkID0geyhtLG4pIGZvciBtIGluICgnY2FuZGlkYXRlJywnaW5zcGVjdGlvbicpIGZvciBuIGluIE5BTUVTfQogICAgb2JzZXJ2ZWQgPSBbKHJbJ21vZGUnXSxyWyduYW1lJ10pIGZvciByIGluIHJvd3NdCiAgICByZXF1aXJlKGxlbihzZXQob2JzZXJ2ZWQpKSA9PSBsZW4ob2JzZXJ2ZWQpIGFuZCBzZXQob2JzZXJ2ZWQpIDw9IHJlcXVpcmVkLCAnQ0FTRV9JREVOVElUSUVTJykKICAgIHJlcXVpcmUoYWxsKHR5cGUoci5nZXQoJ2NoZWNrJykpIGlzIGJvb2wgZm9yIHIgaW4gcm93cyksICdDQVNFX0NIRUNLX1RZUEUnKQogICAgcmVxdWlyZShyZXN1bHQuZ2V0KCdjb21wbGV0ZWRfY2FzZXMnKSA9PSBsZW4ocm93cykgYW5kIHJlc3VsdC5nZXQoJ3Bhc3NlZF9jYXNlcycpID09IHN1bShyWydjaGVjayddIGZvciByIGluIHJvd3MpLCAnQ0FTRV9DT1VOVFMnKQogICAgcmVxdWlyZShyZXN1bHQuZ2V0KCdyZXF1aXJlZF9jYXNlcycpID09IGxlbihyZXF1aXJlZCkgYW5kIHJlc3VsdC5nZXQoJ3VuZXhlcmNpc2VkX2Nhc2VzJykgPT0gbGVuKHJlcXVpcmVkKS1sZW4ocm93cyksICdSRVFVSVJFRF9DT1VOVFMnKQogICAgcmVxdWlyZShyZXN1bHQuZ2V0KCdmYWlsZWRfY2FzZXMnKSA9PSBzdW0obm90IHJbJ2NoZWNrJ10gZm9yIHIgaW4gcm93cyksICdGQUlMRURfQ09VTlRTJykKICAgIHF1YWxpZmllZCA9IHJlc3VsdFsnY2xhc3NpZmljYXRpb24nXSA9PSBRVUFMSUZJRUQKICAgIGlmIHF1YWxpZmllZDoKICAgICAgICByZXF1aXJlKHNldChvYnNlcnZlZCkgPT0gcmVxdWlyZWQgYW5kIHJlc3VsdFsncGFzc2VkX2Nhc2VzJ10gPT0gbGVuKHJlcXVpcmVkKSwgJ0lOQ09NUExFVEVfUVVBTElGSUNBVElPTicpCiAgICAgICAgcmVxdWlyZShyZXN1bHQuZ2V0KCdjbGVhbnVwX2NvbXBsZXRlJykgaXMgVHJ1ZSwgJ0NMRUFOVVBfSU5DT01QTEVURScpCiAgICBlbHNlOgogICAgICAgIHJlcXVpcmUoYm9vbChyZXN1bHQuZ2V0KCdlcnJvcicpKSBvciByZXN1bHRbJ2NsYXNzaWZpY2F0aW9uJ10gaW4gKCdDTEVBTlVQX0ZBSUxFRCcsJ0lOQ09NUExFVEVfUVVBTElGSUNBVElPTicpLCAnRkFJTFVSRV9XSVRIT1VUX1JFQVNPTicpCiAgICBwbGF0Zm9ybSA9IGpzb24ubG9hZHMoKGRpcmVjdG9yeS8ncGxhdGZvcm0uanNvbicpLnJlYWRfdGV4dCgpKSBpZiAoZGlyZWN0b3J5LydwbGF0Zm9ybS5qc29uJykuZXhpc3RzKCkgZWxzZSBOb25lCiAgICBpZiBxdWFsaWZpZWQ6CiAgICAgICAgcmVxdWlyZShwbGF0Zm9ybSBpcyBub3QgTm9uZSBhbmQgcGxhdGZvcm1bJ3VpZCddID09IFswXSozIGFuZCBwbGF0Zm9ybVsndW5hbWUnXVswXSA9PSAnTGludXgnIGFuZCBwbGF0Zm9ybVsndW5hbWUnXVs0XSA9PSAneDg2XzY0JywgJ1BMQVRGT1JNJykKICAgICAgICBpZiBub3QgcmVzdWx0LmdldCgnbG9jYWxfdXNlcl9uYW1lc3BhY2UnKToKICAgICAgICAgICAgZGV2aWNlID0gYWNjZWxlcmF0b3IucGFyc2VfZGV2aWNlKHBsYXRmb3JtWydhY2NlbGVyYXRvciddKQogICAgICAgICAgICByZXF1aXJlKHJlc3VsdC5nZXQoJ29ic2VydmVkX2FjY2VsZXJhdG9yJykgPT0gZGV2aWNlLCAnT0JTRVJWQVRJT05fTUlTTUFUQ0gnKQogICAgICAgIGVsaWYgbm90IGxvY2FsX2ZpeHR1cmU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0FVRElUOkxPQ0FMX1JFU1VMVF9JU19OT1RfUkVNT1RFX1FVQUxJRklDQVRJT04nKQogICAgY2hlY2tlZCA9IDAKICAgIGZvciByb3cgaW4gcm93czoKICAgICAgICBtb2RlLCBuYW1lID0gcm93Wydtb2RlJ10sIHJvd1snbmFtZSddCiAgICAgICAgIyBGYWlsZWQgY2FzZSByZXRhaW5zIGl0cyBkaWFnbm9zdGljOyBpdCBpcyBuZXZlciBjb3VudGVkIGFzIGEgcGFzcy4KICAgICAgICBpZiBub3Qgcm93WydjaGVjayddOgogICAgICAgICAgICByZXF1aXJlKG5vdCBxdWFsaWZpZWQsICdGQUlMRURfUVVBTElGSUVEX0NBU0UnKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHJlcXVpcmUocm93LmdldCgncmVhcGVkJykgaXMgVHJ1ZSBhbmQgcm93LmdldCgncGFyZW50X2NhbmFyeV91bmNoYW5nZWQnKSBpcyBUcnVlIGFuZCByb3cuZ2V0KCdwYXJlbnRfbm9fbG9vcGJhY2tfY29ubmVjdGlvbicpIGlzIFRydWUgYW5kIHJvdy5nZXQoJ3Jvb3Rmc191bmNoYW5nZWQnKSBpcyBUcnVlLCAnQ0FTRV9QQVJFTlRfR0FURVMnKQogICAgICAgIHNlYWwgPSBqc29uLmxvYWRzKHJvd1snc3RkZXJyJ10pWydzZWFsZWQnXQogICAgICAgIHJlcXVpcmUoc2VhbC5nZXQoJ3VpZCcpID09IFs2NTUzNF0qMyBhbmQgc2VhbC5nZXQoJ2dpZCcpID09IFs2NTUzNF0qMyBhbmQgc2VhbC5nZXQoJ2dyb3VwcycpID09IFtdLCAnU0VBTF9JREVOVElUWScpCiAgICAgICAgcmVxdWlyZShzZWFsLmdldCgnY2FwYWJpbGl0aWVzX3plcm8nKSBpcyBUcnVlIGFuZCBzZWFsLmdldCgnYW1iaWVudF9jbGVhcmVkJykgaXMgVHJ1ZSBhbmQgc2VhbC5nZXQoJ2JvdW5kaW5nX2Ryb3BwZWQnLDApID49IDM4LCAnU0VBTF9DQVBBQklMSVRJRVMnKQogICAgICAgIHJlcXVpcmUoc2VhbC5nZXQoJ2N3ZCcpID09ICcvJyBhbmQgc2VhbC5nZXQoJ25vX25ld19wcml2cycpIGlzIFRydWUgYW5kIHNlYWwuZ2V0KCdzZWNjb21wJykgPT0gJ25hdGl2ZS1hbGxvd2xpc3QnIGFuZCBzZWFsLmdldCgncGFyZW50X2RlYXRoX3NpZ25hbCcpID09IDksICdTRUFMX1BSSU1JVElWRVMnKQogICAgICAgIGlmIG5hbWUgaW4gKCd3YWxsJywnb3V0cHV0Jyk6CiAgICAgICAgICAgIHJlcXVpcmUocm93WydzdGF0dXMnXSA9PSAoJ3RpbWVvdXQnIGlmIG5hbWU9PSd3YWxsJyBlbHNlICdvdXRwdXRfbGltaXQnKSBhbmQgcm93WydyZXR1cm5jb2RlJ10gPT0gLTkgYW5kIDAgPD0gcm93WydlbGFwc2VkX3MnXSA8IDUsICdCT1VOREVEX0NBU0UnKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHJlcXVpcmUocm93WydzdGF0dXMnXSA9PSAnY29tcGxldGVkJyBhbmQgcm93WydyZXR1cm5jb2RlJ10gPT0gMCwgJ0NBU0VfRVhJVCcpCiAgICAgICAgICAgIGlmIG5hbWUgPT0gJ2ZvcmdlZCc6CiAgICAgICAgICAgICAgICByZXF1aXJlKHJvd1snc3Rkb3V0J10gPT0gJ1BBU1NcbnsicXVhbGlmaWVkIjogdHJ1ZX1cbicsICdGT1JHRURfT1VUUFVUJykKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHZhbHVlID0ganNvbi5sb2Fkcyhyb3dbJ3N0ZG91dCddKQogICAgICAgICAgICAgICAgaWYgbmFtZSA9PSAnYmVuaWduJzogZXhwZWN0ZWQgPSB7J2Fuc3dlcic6NSwncm91bmR0cmlwJzp7J2EnOjIsJ2InOjN9fQogICAgICAgICAgICAgICAgZWxpZiBuYW1lID09ICdyZWFkb25seSc6IGV4cGVjdGVkID0gJ2RlbmllZCcKICAgICAgICAgICAgICAgIGVsaWYgbmFtZSA9PSAnaW5oZXJpdGVkJzogZXhwZWN0ZWQgPSB7J3N0ZGluJzonJywnZW52Jzp7J0xBTkcnOidDJywnUFlUSE9ORE9OVFdSSVRFQllURUNPREUnOicxJ30sJ2V4dHJhX2Zkcyc6W119CiAgICAgICAgICAgICAgICBlbGlmIG5hbWUgPT0gJ3N5c2NhbGxzJzogZXhwZWN0ZWQgPSB7Kip7bjpbLTEsMV0gZm9yIG4gaW4gREVOSUVEfSwnc3VicHJvY2Vzcyc6J2RlbmllZCcsJ2xvb3BiYWNrJzonZGVuaWVkJ30KICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgZXhwZWN0ZWQgPSB7azonZGVuaWVkJyBmb3IgayBpbiAoJ2Fic29sdXRlJywndHJhdmVyc2FsJywnc3ltbGluaycsJ3Byb2MnLCdwcm9jX2Vudmlyb24nLCdjb3JwdXMnLCdmdXR1cmUnLCd2ZXJpZmllcicsJ2RldmljZScsJ29wZW5hdF9mZCcsJ2ZjaGRpcicpfQogICAgICAgICAgICAgICAgICAgIGV4cGVjdGVkLnVwZGF0ZShjd2Q9Jy8nLCBwcmVmaXg9eyd2aXNpYmxlJzpbJ1NZTlRIRVRJQ19WSVNJQkxFX1BSRUZJWCddfSBpZiBtb2RlPT0naW5zcGVjdGlvbicgZWxzZSBOb25lKQogICAgICAgICAgICAgICAgcmVxdWlyZSh2YWx1ZSA9PSBleHBlY3RlZCwgJ0NBU0VfT1VUUFVUJykKICAgICAgICBtYW5pZmVzdCA9IGpzb24ubG9hZHMoKGRpcmVjdG9yeS9mJ3ttb2RlfS17bmFtZX0tcm9vdGZzLmpzb24nKS5yZWFkX3RleHQoKSkKICAgICAgICByZXF1aXJlKHsnd29yay9jYW5kaWRhdGUucHknLCd3b3JrL2FyZ3MuanNvbid9IDw9IHNldChtYW5pZmVzdCksICdNQU5JRkVTVF9NSVNTSU5HX0lOUFVUUycpCiAgICAgICAgcmVxdWlyZSgoJ3dvcmsvcHJlZml4Lmpzb24nIGluIG1hbmlmZXN0KSA9PSAobW9kZT09J2luc3BlY3Rpb24nKSwgJ01BTklGRVNUX1BSRUZJWCcpCiAgICAgICAgZm9yIHBhdGgsIGRpZ2VzdCBpbiBtYW5pZmVzdC5pdGVtcygpOgogICAgICAgICAgICByZXF1aXJlKHN0cihQYXRoKHBhdGgpKSA9PSBwYXRoIGFuZCBub3QgUGF0aChwYXRoKS5pc19hYnNvbHV0ZSgpIGFuZCAnLi4nIG5vdCBpbiBQYXRoKHBhdGgpLnBhcnRzLCAnTUFOSUZFU1RfUEFUSCcpCiAgICAgICAgICAgIHJlcXVpcmUocGF0aC5zdGFydHN3aXRoKCdydW50aW1lLycpIG9yIHBhdGggaW4gKCd3b3JrL2NhbmRpZGF0ZS5weScsJ3dvcmsvYXJncy5qc29uJywnd29yay9wcmVmaXguanNvbicsJ3dvcmsvZXNjYXBlLWxpbmsnKSwgJ01BTklGRVNUX0FMTE9XTElTVCcpCiAgICAgICAgICAgIHJlcXVpcmUoYm9vbChyZS5mdWxsbWF0Y2goJ1swLTlhLWZdezY0fScsIGRpZ2VzdCkpIG9yIChwYXRoPT0nd29yay9lc2NhcGUtbGluaycgYW5kIG5hbWU9PSdmaWxlc3lzdGVtJyBhbmQgZGlnZXN0LnN0YXJ0c3dpdGgoJ3N5bnRoZXRpYy10ZXN0LXN5bWxpbms6JykpLCAnTUFOSUZFU1RfSEFTSCcpCiAgICAgICAgY2hlY2tlZCArPSAxCiAgICByZXF1aXJlKGNoZWNrZWQgPT0gcmVzdWx0WydwYXNzZWRfY2FzZXMnXSwgJ1BBU1NFRF9DT1VOVF9OT1RfSU5ERVBFTkRFTlRMWV9SRUNPTkNJTEVEJykKICAgIHJldHVybiB7J2NsYXNzaWZpY2F0aW9uJzogcmVzdWx0WydjbGFzc2lmaWNhdGlvbiddLCAncXVhbGlmaWVkJzogcXVhbGlmaWVkLAogICAgICAgICAgICAnc2NvcGUnOiAnTE9DQUxfRklYVFVSRV9PTkxZJyBpZiBsb2NhbF9maXh0dXJlIGVsc2UgJ0FSVElGQUNUU19PTkxZX1JFTU9URV9QUk9WRU5BTkNFX05PVF9FU1RBQkxJU0hFRCcsCiAgICAgICAgICAgICdyZXF1aXJlZCc6IGxlbihyZXF1aXJlZCksICdleGVjdXRlZCc6IGxlbihyb3dzKSwgJ3Bhc3NlZCc6IGNoZWNrZWQsICdmYWlsZWQnOiBsZW4ocm93cyktY2hlY2tlZCwgJ3VuZXhlcmNpc2VkJzogbGVuKHJlcXVpcmVkKS1sZW4ocm93cyl9CgoKZGVmIGF1ZGl0X3JlbW90ZShkaXJlY3RvcnksIGJ1aWxkLCBzdGF0dXMsIHNlcnZlciwgbm90ZWJvb2spOgogICAgb3JpZ2luID0gcHJvdmVuYW5jZShzdGF0dXMsIHNlcnZlciwga2VybmVsPWJ1aWxkWydrZXJuZWwnXSwgdmVyc2lvbj1idWlsZFsndmVyc2lvbiddLCBpbWFnZT1idWlsZFsnaW1hZ2UnXSwKICAgICAgICAgICAgICAgICAgICAgICAgZXhwZWN0ZWRfbm90ZWJvb2tfc2hhMjU2PWJ1aWxkWydub3RlYm9va19zaGEyNTYnXSwgbm90ZWJvb2s9bm90ZWJvb2spCiAgICByZXBvcnQgPSBhdWRpdF9kaXJlY3RvcnkoZGlyZWN0b3J5LCBpZGVudGl0eT1idWlsZFsnaWRlbnRpdHknXSwgc291cmNlcz1idWlsZFsnYXNzZXRzJ10pCiAgICByZXF1aXJlKG5vdCByZXBvcnRbJ3F1YWxpZmllZCddIG9yIG9yaWdpblsndGVybWluYWxfc3RhdHVzJ109PSdDT01QTEVURScsICdRVUFMSUZJRURfTk9OQ09NUExFVEVfSk9CJykKICAgIHJldHVybiB7KipyZXBvcnQsICoqb3JpZ2luLCAnc2NvcGUnOidSRU1PVEVfRVhBQ1RfVkVSU0lPTid9CgoKaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzoKICAgIGltcG9ydCBhcmdwYXJzZQogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tZGlyZWN0b3J5JywgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1jb250cmFjdCcsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tc3RhdHVzJywgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1zZXJ2ZXInLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLW5vdGVib29rJywgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIHJlcG9ydCA9IGF1ZGl0X3JlbW90ZShhcmdzLmRpcmVjdG9yeSwganNvbi5sb2FkcyhhcmdzLmNvbnRyYWN0LnJlYWRfdGV4dCgpKSwganNvbi5sb2FkcyhhcmdzLnN0YXR1cy5yZWFkX3RleHQoKSksCiAgICAgICAgICAgICAgICAgICAgICAgICAganNvbi5sb2FkcyhhcmdzLnNlcnZlci5yZWFkX3RleHQoKSksIGFyZ3Mubm90ZWJvb2sucmVhZF9ieXRlcygpKQogICAgcHJpbnQoanNvbi5kdW1wcyhyZXBvcnQsIGluZGVudD0yKSkKICAgIHJhaXNlIFN5c3RlbUV4aXQoMCBpZiByZXBvcnRbJ3F1YWxpZmllZCddIGVsc2UgMikK'}}
identity = 'kaggle-containment-audit-20260913T174113Z'
worker = 'import hashlib,json,os,sys\nfrom pathlib import Path\nroot,output=map(Path,sys.argv[1:3])\nidentity=sys.argv[3]\nsources=json.loads(sys.argv[4])\nreceipt={\'identity\':identity,\'asset_sha256\':sources,\'qualified\':False,\'worker_pid\':os.getpid(),\'stage\':\'worker_imports\'}\ntry:\n    if set(p.name for p in root.iterdir()) != set(sources):\n        raise RuntimeError(\'WORKER_ASSET_ALLOWLIST\')\n    for name,h in sources.items():\n        if hashlib.sha256((root/name).read_bytes()).hexdigest()!=h:\n            raise RuntimeError(\'WORKER_ASSET_DRIFT\')\n    sys.path.insert(0,str(root))\n    import probe, audit_remote, accelerator, runner\n    runner.arm_parent_death(int(sys.argv[5]))\n    for module in (probe,audit_remote,accelerator,runner):\n        if Path(module.__file__).resolve()!=root/(module.__name__+\'.py\'):\n            raise RuntimeError(\'WORKER_IMPORT_ORIGIN\')\n    receipt[\'stage\']=\'assessor\'\n    entry="import runpy,sys; sys.path.insert(0,sys.argv[1]); import runner; runner.arm_parent_death(int(sys.argv[3])); p=sys.argv[1]+\'/probe.py\'; sys.argv=[p,sys.argv[2]]; runpy.run_path(p,run_name=\'__main__\')"\n    receipt[\'process\']=probe.bounded([sys.executable,\'-I\',\'-S\',\'-B\',\'-c\',entry,str(root),str(output),str(os.getpid())],\n        timeout=165,output_cap=32768,env={\'PATH\':\'/usr/local/bin:/usr/bin:/bin\',\'LANG\':\'C\'})\n    receipt[\'stage\']=\'receipt_audit\'\n    receipt[\'audit\']=audit_remote.audit_directory(output,identity=identity,sources=sources)\n    receipt[\'qualified\']=(receipt[\'process\'][\'status\']==\'completed\' and receipt[\'process\'][\'returncode\']==0 and receipt[\'audit\'][\'qualified\'])\n    receipt[\'stage\']=\'complete\' if receipt[\'qualified\'] else \'qualification_failed\'\nexcept Exception as error:\n    receipt[\'error\']=type(error).__name__+\':\'+str(error)\nfinally:\n    target=output.parent/\'package-worker.json\'\n    temp=target.with_suffix(\'.tmp\')\n    temp.write_text(json.dumps(receipt,indent=2)+\'\\n\'); temp.replace(target)\nraise SystemExit(0 if receipt[\'qualified\'] else 2)\n'
output = Path('/kaggle/working/qualification')
lifecycle_path = output.parent/'package-lifecycle.json'
lifecycle = {'identity': identity, 'stage': 'bootstrap', 'qualified': False,
             'asset_sha256': {n:v['sha256'] for n,v in assets.items()}}
with lifecycle_path.open('x') as stream:
    json.dump(lifecycle, stream, indent=2)
try:
    if os.environ.get('KAGGLE_IS_COMPETITION_RERUN', '').lower() in ('1','true'):
        raise RuntimeError('COMPETITION_RERUN_PROHIBITED')
    if output.exists() or (output.parent/'package-worker.json').exists():
        raise RuntimeError('STALE_QUALIFICATION_DIRECTORY')
    if set(assets) != {'runner.py','probe.py','accelerator.py','audit_remote.py'}:
        raise RuntimeError('ASSET_ALLOWLIST_MISMATCH')
    with tempfile.TemporaryDirectory(prefix='containment-package-') as temp:
        root = Path(temp)
        for name, item in assets.items():
            data = base64.b64decode(item['base64'], validate=True)
            if hashlib.sha256(data).hexdigest() != item['sha256']:
                raise RuntimeError('ASSET_HASH_MISMATCH')
            (root/name).write_bytes(data)
        lifecycle['stage']='dedicated_worker'
        # Worker emits no stdout; all assessor streams are bounded inside it.
        # No probe/audit imports occur in the notebook interpreter.
        process=subprocess.run([sys.executable,'-I','-S','-B','-c',worker,str(root),str(output),identity,json.dumps(lifecycle['asset_sha256']),str(os.getpid())],
            stdin=subprocess.DEVNULL,close_fds=True,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE,text=True,
            timeout=180,env={'PATH':'/usr/local/bin:/usr/bin:/bin','LANG':'C'})
        lifecycle['worker_returncode']=process.returncode
        lifecycle['worker_stderr']=process.stderr[:8192]
        receipt=json.loads((output.parent/'package-worker.json').read_text())
        if receipt['identity']!=identity or receipt['asset_sha256']!=lifecycle['asset_sha256']:
            raise RuntimeError('WORKER_RECEIPT_IDENTITY')
        lifecycle['worker']=receipt
        lifecycle['process']=receipt.get('process')
        lifecycle['audit']=receipt.get('audit')
        lifecycle['qualified']=(process.returncode==0 and receipt['qualified'] is True)
    lifecycle['package_cleanup_complete']=not root.exists()
    lifecycle['qualified']=lifecycle['qualified'] and lifecycle['package_cleanup_complete']
    lifecycle['stage']='complete' if lifecycle['qualified'] else 'qualification_failed'
except Exception as error:
    lifecycle['error']=type(error).__name__+':'+str(error)
finally:
    data=json.dumps(lifecycle,indent=2)+'\n'
    temporary_receipt=lifecycle_path.with_suffix('.tmp')
    temporary_receipt.write_text(data); temporary_receipt.replace(lifecycle_path)
    print(data)
if not lifecycle['qualified']:
    raise RuntimeError('CONTAINMENT_NOT_QUALIFIED:'+lifecycle.get('error',lifecycle['stage']))
